# Lesson 3.3: What Is Generative AI?

*Module 3 · Introduction to Generative AI*

> Every AI before 2022 was a judge — it classified data as "cat" or "dog." Generative AI is fundamentally different. It creates new things . This lesson covers how it works, why it works now, what breaks, how to customize it (prompting vs fine-tuning vs RAG), and where the field is heading with agents.

`Beginner Friendly` · `8 Concepts` · `Gemini API` · `120 min`

---

## About this notebook

This notebook contains the **9 runnable code examples** from the published lesson `lesson-3.3.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Two Types of AI — The Judge vs The Chef — `discriminative_ai.py`
2. Step 1 — Two Types of AI — The Judge vs The Chef — `generative_ai.py`
3. Step 2 — How Does It Work? — The Next-Word Game — `temperature_demo.py`
4. Step 3 — The Universe of Generative AI — It's Not Just ChatGPT — `genai_universe.py`
5. Step 4 — Why Now? — Three Breakthroughs That Changed Everything — `scale_effect.py`
6. Step 5 — What Can Go Wrong? — The Limitations You Must Know — `limitations_demo.py`
7. Step 6 — Three Ways to Customize — Prompting → RAG → Fine-Tuning — `customization_spectrum.py`
8. Step 7 — Context Windows, Tokens & the Model Landscape — `model_landscape.py`
9. Step 8 — Beyond Chat — RAG & Agents — `rag_and_agents.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai scikit-learn


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Two Types of AI — The Judge vs The Chef

The most important distinction in all of machine learning.

#### See the difference in code

**`discriminative_ai.py`** · _Block 1 of 9_

DISCRIMINATIVE AI — The Judge — Input: movie review → Output: "positive" or "negative"


In [ ]:
# =============================================
# DISCRIMINATIVE AI — The Judge
# Input: movie review → Output: "positive" or "negative"
# It classifies. It NEVER creates new text.
# =============================================

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Training data
reviews = [
    "This movie was absolutely fantastic! Loved every minute.",
    "Terrible film. Waste of time and money.",
    "A masterpiece! The acting was incredible.",
    "Boring and predictable. Would not recommend.",
    "One of the best films I've ever seen!",
    "Awful screenplay. The plot made no sense.",
]
labels = ["positive", "negative", "positive", "negative", "positive", "negative"]

# Train
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(reviews)
model = MultinomialNB()
model.fit(X, labels)

# Classify a NEW review
new_review = "The movie was engaging and well-directed."
prediction = model.predict(vectorizer.transform([new_review]))[0]

print(f"Review:  '{new_review}'")
print(f"AI says: {prediction}")
print(f"Can it WRITE a review? ❌ No. It only judges.")


#### See the difference in code

**`generative_ai.py`** · _Block 2 of 9_

GENERATIVE AI — The Creator — Input: a prompt → Output: NEW text that never existed


In [ ]:
# =============================================
# GENERATIVE AI — The Creator
# Input: a prompt → Output: NEW text that never existed
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# Task 1: CREATE a movie review (not classify — CREATE)
resp = model.generate_content(
    "Write a 3-sentence review for a fictional Bollywood "
    "sci-fi film called 'Mars Wala Love'."
)
print("Generated review:")
print(f"  {resp.text}\n")

# Task 2: CREATE code
resp = model.generate_content("Write a Python function to check if a number is prime.")
print("Generated code:")
print(f"  {resp.text}\n")

# Task 3: CREATE an analogy
resp = model.generate_content("Explain WiFi using the analogy of a chai stall in a busy market.")
print("Generated analogy:")
print(f"  {resp.text}")


> **What just happened?** Discriminative AI took a review and output ONE WORD: "positive." That's its entire capability. Generative AI took short instructions and produced an entire movie review, a Python function, and a creative analogy — none of which existed before. The key insight: Generative AI can do everything discriminative AI does (classification is just generating "positive") — but discriminative AI cannot create.


### Step 2 · How Does It Work? — The Next-Word Game

The entire magic of Large Language Models in one concept.

#### Temperature — the creativity dial

**`temperature_demo.py`** · _Block 3 of 9_

TEMPERATURE EXPERIMENT — Same prompt, different creativity levels.


In [ ]:
# =============================================
# TEMPERATURE EXPERIMENT
# Same prompt, different creativity levels.
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

prompt = "Continue in 1 sentence: 'The robot opened its eyes for the first time and saw'"

for temp in [0.0, 0.5, 1.0, 1.5]:
    model = genai.GenerativeModel("gemini-2.0-flash",
        generation_config={"temperature": temp, "max_output_tokens": 40})
    resp = model.generate_content(prompt)
    labels = {0.0: "📏 Predictable", 0.5: "⚖️ Balanced",
              1.0: "🎨 Creative", 1.5: "🌪️ Wild"}
    print(f"  temp={temp} {labels[temp]}:")
    print(f"    {resp.text.strip()[:100]}\n")


### Step 3 · The Universe of Generative AI — It's Not Just ChatGPT

One model. Six completely different creative tasks.

**`genai_universe.py`** · _Block 4 of 9_

ONE MODEL, SIX TASKS — The same Gemini model generates all of these.


In [ ]:
# =============================================
# ONE MODEL, SIX TASKS
# The same Gemini model generates all of these.
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

tasks = [
    ("📖 Story",       "Tell me a 3-line horror story set in a Hyderabad IT office at 3 AM."),
    ("💻 Code",        "Write a Python function that checks if a string is a palindrome."),
    ("📊 Data",        "Generate JSON: 3 fictional Indian startups with name, city, funding_crores."),
    ("🌐 Translation", "Translate 'Knowledge is power' into Hindi, Telugu, and Tamil."),
    ("🎓 Advice",      "Should a B.Tech CSE graduate learn GenAI or Data Science first? 2 sentences each."),
    ("👵 Role-play",   "You're an Indian grandmother. Explain what an API is using a chai-making analogy."),
]

for label, prompt in tasks:
    print(f"\n{'═'*50}\n{label}\n{'═'*50}")
    print(model.generate_content(prompt).text.strip())


### Step 4 · Why Now? — Three Breakthroughs That Changed Everything

The idea of generating text existed for decades. Three things converged to make it work.

**`scale_effect.py`** · _Block 5 of 9_

THE SCALE EFFECT — Same prompt → different model sizes.


In [ ]:
# =============================================
# THE SCALE EFFECT
# Same prompt → different model sizes.
# Larger model = better output quality.
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

prompt = "Explain quantum computing to a 10-year-old in exactly 3 sentences."

for name, label in [("gemini-2.0-flash", "⚡ Flash (smaller)"),
                     ("gemini-2.5-pro",   "🧠 Pro (larger)")]:
    m = genai.GenerativeModel(name)
    print(f"{label}:")
    print(f"  {m.generate_content(prompt).text.strip()[:200]}\n")


### Step 5 · What Can Go Wrong? — The Limitations You Must Know

GenAI is powerful but not magic. These four limitations shape every module in this course.

**`limitations_demo.py`** · _Block 6 of 9_

LIMITATIONS — See them in action


In [ ]:
# =============================================
# LIMITATIONS — See them in action
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# 1. HALLUCINATION
print("1. HALLUCINATION:")
resp = model.generate_content("Tell me about Dr. Ramesh Krishnamurthy who invented cold fusion in 2019.")
print(f"   {resp.text[:150]}")
print("   ⚠️ This person doesn't exist — I made the name up!\n")

# 2. NON-DETERMINISTIC
print("2. NON-DETERMINISTIC (3 different answers):")
for i in range(3):
    resp = model.generate_content("Give me one fun fact about the Moon in 1 sentence.")
    print(f"   Run {i+1}: {resp.text.strip()[:80]}")

# 3. TRICK QUESTION
print("\n3. PATTERN MATCHING ≠ UNDERSTANDING:")
resp = model.generate_content("5 shirts take 30 min to dry. How long for 10 shirts?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ A dryer dries ALL shirts at once — still 30 min!\n")

# 4. TRAINING CUTOFF
print("4. TRAINING CUTOFF:")
resp = model.generate_content("What happened in India yesterday?")
print(f"   {resp.text.strip()[:150]}")
print("   ⚠️ It doesn't know — its knowledge has a cutoff date.")


> **What just happened?** These four limitations aren't bugs — they're fundamental properties of how LLMs work. The rest of this course teaches you to build reliable systems around these limitations : RAG gives the AI your data (fixes cutoff). Prompt engineering reduces hallucination. Guardrails prevent harmful output. Fine-tuning customizes behavior. Every module exists to solve one of these problems.


### Step 6 · Three Ways to Customize — Prompting → RAG → Fine-Tuning

You don't build LLMs from scratch. You customize existing ones. There are three ways, each with different cost and control.

**`customization_spectrum.py`** · _Block 7 of 9_

THREE WAYS TO CUSTOMIZE AN LLM — 1. Prompting: cheapest, fastest, least control


In [ ]:
# =============================================
# THREE WAYS TO CUSTOMIZE AN LLM
# 1. Prompting: cheapest, fastest, least control
# 2. RAG: add your own knowledge at query time
# 3. Fine-tuning: retrain on your data
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# ── 1. PROMPTING — Change behavior with instructions ──
# Free, instant. Limited by what the model already knows.
print("1. PROMPTING (zero cost):")
resp = model.generate_content(
    "You are a senior Python developer. Review this code: "
    "def add(a, b): return a + b\n"
    "Give 3 improvement suggestions."
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 2. RAG — Inject your own documents ──
# Your data goes into the CONTEXT, not the model weights.
print("2. RAG (your data + model = accurate answers):")
company_docs = """
NETSETOS REFUND POLICY (2024):
- Full refund within 7 days of purchase.
- 50% refund within 30 days.
- No refund after 30 days.
- Contact: [email protected]
"""
resp = model.generate_content(
    f"Based ONLY on this document:\n{company_docs}\n\n"
    f"Question: Can I get a refund after 15 days?"
)
print(f"   {resp.text.strip()[:150]}\n")

# ── 3. FINE-TUNING — Retrain the model ──
# Changes the model's weights. Expensive but powerful.
print("3. FINE-TUNING (changes model behavior permanently):")
print("   Example: Train Gemini on 10,000 customer support tickets")
print("   Result: Model learns your company's tone, policies, jargon")
print("   Cost: ~$50-500 + 1-4 hours of training time")
print("   When: Only after prompting + RAG aren't enough (Module 9)\n")

print("Decision framework:")
print("  Start with prompting (free, 5 min)")
print("  Add RAG if model needs YOUR data (Module 8)")
print("  Fine-tune only if behavior must change fundamentally (Module 9)")


> **What just happened?** Prompting changed the model's behavior with instructions — zero cost, instant. RAG injected your company's refund policy into the context — the model answered accurately from YOUR data without any training. Fine-tuning would permanently change the model's weights — expensive but powerful. 80% of production GenAI apps use prompting + RAG. Fine-tuning is the last resort. This "customization spectrum" is the most important architectural decision you'll make. Module 5 covers prompting, Module 8 covers RAG, Module 9 covers fine-tuning.


### Step 7 · Context Windows, Tokens & the Model Landscape

The context window is the model's working memory. Everything it reads and writes must fit inside it.

**`model_landscape.py`** · _Block 8 of 9_

THE 2025 MODEL LANDSCAPE — Context windows, costs, and when to use each


In [ ]:
# =============================================
# THE 2025 MODEL LANDSCAPE
# Context windows, costs, and when to use each
# =============================================

models = [
    # (Name, Provider, Context, Input$/M, Open-source?)
    ("Gemini 2.0 Flash",    "Google",    1_000_000,  0.15,  False),
    ("GPT-4o",              "OpenAI",      128_000,  2.50,  False),
    ("Claude 3.5 Sonnet",   "Anthropic",   200_000,  3.00,  False),
    ("LLaMA 3.1 70B",      "Meta",        128_000,  0.00,  True),
    ("Mistral Large",      "Mistral",     128_000,  2.00,  False),
    ("DeepSeek V3",        "DeepSeek",    128_000,  0.27,  True),
]

print(f"{'Model':<22s} {'Provider':<12s} {'Context':>10s} {'$/M Input':>10s} {'Open?':>6s}")
print("-" * 65)
for name, provider, ctx, cost, oss in models:
    ctx_str = f"{ctx//1000}K"
    cost_str = "FREE" if cost == 0 else f"${cost:.2f}"
    oss_str = "✅ Yes" if oss else "No"
    print(f"  {name:<20s} {provider:<12s} {ctx_str:>10s} {cost_str:>10s} {oss_str:>6s}")

print(f"""
Key decisions:
  Gemini Flash  → Default choice. Cheapest, 1M context, multimodal.
  GPT-4o        → When you need OpenAI ecosystem (function calling, assistants).
  Claude Sonnet → Best for writing, long docs, code generation.
  LLaMA/DeepSeek → Self-host for privacy, no API costs, full control.

Open-source = you can run it on YOUR servers.
Closed-source = API only, vendor controls the model.
Module 13 covers self-hosting with Ollama and vLLM.
""")


> **What just happened?** The model landscape in 2025 has clear trade-offs: Gemini Flash is cheapest with the largest context (1M tokens = entire books). GPT-4o has the best ecosystem. Claude excels at writing. LLaMA/DeepSeek are free and open-source — you can run them on your own servers for privacy. The context window determines what fits: with 1M tokens, you can feed an entire codebase. With 4K tokens, you can barely fit a conversation. In production: use multi-provider architecture (Module 7) — Gemini primary, OpenAI fallback, Claude for writing. Never depend on one provider.


### Step 8 · Beyond Chat — RAG & Agents

LLMs alone have limitations. RAG gives them knowledge. Agents give them actions. Together, they're the future of AI applications.

**`rag_and_agents.py`** · _Block 9 of 9_

RAG: Give the LLM YOUR knowledge — AGENTS: Give the LLM TOOLS


In [ ]:
# =============================================
# RAG: Give the LLM YOUR knowledge
# AGENTS: Give the LLM TOOLS
# =============================================

import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
model = genai.GenerativeModel("gemini-2.0-flash")

# ── 1. PLAIN LLM — Limited to training data ──
print("1. PLAIN LLM (no knowledge of your company):")
resp = model.generate_content("What is Netsetos's refund policy?")
print(f"   {resp.text.strip()[:120]}")
print("   ⚠️ It doesn't know — will hallucinate or say 'I don't know'\n")

# ── 2. RAG — Fetch docs, stuff into context ──
print("2. RAG (Retrieval-Augmented Generation):")
print("   Step 1: User asks 'What's the refund policy?'")
print("   Step 2: System searches your document database")
print("   Step 3: Relevant policy document is RETRIEVED")
print("   Step 4: Document is stuffed into the CONTEXT")
print("   Step 5: LLM generates answer FROM the document")
print("   → Accurate, grounded, no hallucination!")
print("   → Module 8 builds this complete pipeline\n")

# ── 3. AGENTS — LLM + Tools ──
print("3. AGENTS (LLM + Tools = Actions):")
print("   User: 'Book me a flight to Mumbai tomorrow under ₹5000'")
print("   Agent thinks: I need to search flights → compare prices → book")
print("   Step 1: Calls flight search API (tool)")
print("   Step 2: Filters results under ₹5000 (reasoning)")
print("   Step 3: Calls booking API (tool)")
print("   Step 4: Confirms with user (human-in-the-loop)")
print("   → The LLM didn't just answer — it ACTED!")
print("   → Module 11 builds multi-step agents\n")

print("The evolution of GenAI applications:")
print("  Chat (Module 3-3)  →  RAG (Module 8)  →  Agents (Module 11)")
print("  'Answer from memory'  'Answer from docs'  'Take actions'")


> **What just happened?** Three levels of GenAI applications: (1) Plain LLM — answers from training data only (limited, hallucination-prone). (2) RAG — retrieves YOUR documents and uses them as context (accurate, grounded, the #1 pattern in production). (3) Agents — LLM + tools (search, code execution, APIs) = the model takes multi-step ACTIONS, not just answers. This is the course roadmap: Modules 3-7 build chat apps, Module 8 adds RAG, Modules 10-11 build agents. By Module 16, you'll combine all three into a production capstone.


---

## Wrap-up

You've walked through all 9 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
